## Notebook 03 — Exploration non supervisée


### <u> 1. Étapes du notebook </u>

1. **Préparation** — chargement du train, imputation et mise à l'échelle pour l'analyse
2. **Réduction de dimension (PCA)** — variance expliquée, projection 2D colorée par le top 10
3. **Détection d'anomalies (GMM)** — lignes les moins probables et lecture métier
4. **Clustering (K-Means)** — choix du nombre de groupes, profils obtenus
5. **Clusters et cible** — taux de top 10 par cluster
6. **Conclusions** — ce que l'analyse confirme pour la modélisation


<br/>

### <u> 2. Principes </u>

**Train uniquement** — saisons 2006 à 2021, le test reste de côté

**Sans la cible** — PCA, anomalies et clusters sont calculés sans top10 ni la position ; la cible sert seulement à interpréter les résultats

**Transformations internes** — imputation et mise à l'échelle faites dans ce notebook pour l'analyse, rien n'est sauvegardé

<br/>

### <u> 3. Résultat attendu </u>

- **Redondances** — confirmées ou non par la PCA
- **Anomalies** — situations réelles à conserver, ou erreurs à corriger
- **Profils** — clusters de pilotes liés ou non au top 10

---------------------------------------------------------------------------------------------

In [ ]:
# ÉTAPE 1 : PRÉPARATION

# Imports
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# Rappel Notebook 2 (chargement et découpage)
data = pd.read_csv("../data/processed-data/f1_table.csv")
train = data[data["year"] <= 2021]
NUMERICAL_FEATURE = ["driver_avg_position_last5", "driver_dnf_rate_last10", "driver_season_points_before_race","driver_career_starts", "driver_avg_position_on_circuit","team_avg_position_last5", "team_dnf_rate_last10", "team_season_points_before_race","grid_position_clean", "grid_gap_to_teammate", "quali_gap_to_best_q1_percent", "quali_position"]


# Mise de côté du top 10
top10 = train["top10"]


# Pipeline : imputation par la médiane puis mise à l'échelle
pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler", StandardScaler())])
X = pipeline.fit_transform(train[NUMERICAL_FEATURE])


# Vérification
print(f"Dimensions : {X.shape}")                       
print(f"Valeurs manquantes : {np.isnan(X).sum()}")      
print(f"Moyennes : {X.mean(axis=0).round(2)}")     
print(f"Écarts-types : {X.std(axis=0).round(2)}")       

In [ ]:
# ÉTAPE 2 : RÉDUCTION DE DIMENSION

# Imports
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


# PCA avec toutes les composantes 
pca = PCA()
pca.fit(X)
variance_cumulee = np.cumsum(pca.explained_variance_ratio_)
print(f"Variance par composante : {pca.explained_variance_ratio_.round(3)}")
print(f"Composantes pour 95 % : {np.argmax(variance_cumulee >= 0.95) + 1}")


# Tracé 
plt.plot(range(1, len(variance_cumulee) + 1), variance_cumulee, marker="o")
plt.axhline(y=0.95, color="r", linestyle="--", label="95 %")
plt.xlabel("Nombre de composantes")
plt.ylabel("Variance cumulée")
plt.legend()
plt.show()


# Poids de chaque features pour les 8 première composnates 
poids = pd.DataFrame(pca.components_[:8].T, index=NUMERICAL_FEATURE, columns=["Composante 1", "Composante 2", "Composante 3", "Composante 4", "Composante 5", "Composante 6", "Composante 7", "Composante 8"])
display(poids.round(2))